# CICIDS2017 Data Preprocessing

This notebook processes the raw CSV files from the CICIDS2017 dataset.

**Steps:**
1. Merge all CSV files.
2. Clean column headers.
3. Encode Labels (Benign=0, Attack=1).
4. Clean Data (Remove Inf/NaN, Duplicates).
5. Stratified Sampling (10%).
6. Save processed data.

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
from sklearn.model_selection import train_test_split
import time

# Start Timer for Computational Cost Metric
start_time = time.time()
print("Timer started...")

In [ ]:
# 1. Load and Merge Data
# Explicitly define column names to match the CSV headers and ensure consistency
# Note: 'Fwd Header Length' appears twice in the original dataset headers.
# We rename the second occurrence to 'Fwd Header Length.1' to avoid a ValueError in pd.read_csv.
columns = [
    "Destination Port", "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
    "Total Length of Fwd Packets", "Total Length of Bwd Packets", "Fwd Packet Length Max",
    "Fwd Packet Length Min", "Fwd Packet Length Mean", "Fwd Packet Length Std",
    "Bwd Packet Length Max", "Bwd Packet Length Min", "Bwd Packet Length Mean",
    "Bwd Packet Length Std", "Flow Bytes/s", "Flow Packets/s", "Flow IAT Mean",
    "Flow IAT Std", "Flow IAT Max", "Flow IAT Min", "Fwd IAT Total", "Fwd IAT Mean",
    "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min", "Bwd IAT Total", "Bwd IAT Mean",
    "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min", "Fwd PSH Flags", "Bwd PSH Flags",
    "Fwd URG Flags", "Bwd URG Flags", "Fwd Header Length", "Bwd Header Length",
    "Fwd Packets/s", "Bwd Packets/s", "Min Packet Length", "Max Packet Length",
    "Packet Length Mean", "Packet Length Std", "Packet Length Variance", "FIN Flag Count",
    "SYN Flag Count", "RST Flag Count", "PSH Flag Count", "ACK Flag Count", "URG Flag Count",
    "CWE Flag Count", "ECE Flag Count", "Down/Up Ratio", "Average Packet Size",
    "Avg Fwd Segment Size", "Avg Bwd Segment Size", "Fwd Header Length.1", "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate", "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate", "Subflow Fwd Packets", "Subflow Fwd Bytes", "Subflow Bwd Packets",
    "Subflow Bwd Bytes", "Init_Win_bytes_forward", "Init_Win_bytes_backward",
    "act_data_pkt_fwd", "min_seg_size_forward", "Active Mean", "Active Std", "Active Max",
    "Active Min", "Idle Mean", "Idle Std", "Idle Max", "Idle Min", "Label"
]

# The files are located in 'MachineLearningCSV/MachineLearningCVE/'
path = os.path.join("MachineLearningCSV", "MachineLearningCVE")
all_files = glob.glob(os.path.join(path, "*.csv"))

print(f"Found {len(all_files)} files to merge.")

df_list = []
for filename in all_files:
    print(f"Reading {os.path.basename(filename)}...")
    # Use 'names' to enforce column names, header=0 skips the file's header row
    # low_memory=False to process large chunks efficiently, but might warn on mixed types
    df_iter = pd.read_csv(filename, names=columns, header=0, index_col=None)
    df_list.append(df_iter)

df = pd.concat(df_list, axis=0, ignore_index=True)
print(f"Combined DataFrame Shape: {df.shape}")

In [ ]:
# 2. Clean Column Headers
# The provided list contains 'Fwd Header Length' twice, so Pandas may have created a duplicate.
# We remove the duplicate column (usually labeled with .1) if it exists.

# Strip whitespace just in case any slipped in
df.columns = df.columns.str.strip()

# Check for duplicate 'Fwd Header Length'
if "Fwd Header Length.1" in df.columns:
    print("Dropping duplicate column 'Fwd Header Length.1'")
    df.drop(columns=["Fwd Header Length.1"], inplace=True)

print("Final Columns List:")
print(df.columns.tolist())

In [ ]:
# 3. Handle Labels (Binary Classification)
# Map 'BENIGN' -> 0, Everything else -> 1

def map_label(label):
    if label == 'BENIGN':
        return 0
    else:
        return 1

df['Label'] = df['Label'].apply(map_label)

print("Label distribution before sampling:")
print(df['Label'].value_counts())

In [ ]:
# 4. Data Cleaning
# Check for Infinite values and NaN
# Replace infinity with Nan then drop

print("Cleaning Infinite and NaN values...")
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Drop Duplicates
print(f"Shape before dropping duplicates: {df.shape}")
df.drop_duplicates(inplace=True)
print(f"Shape after dropping duplicates: {df.shape}")

In [ ]:
# 5. Sampling (10%)
# Use stratified sampling to preserve class balance
# train_size=0.1 means we choose 10% of the data to keep in 'df_sample'

print("Sampling 10% of data...")
df_sample, _ = train_test_split(df, train_size=0.1, stratify=df['Label'], random_state=42)

print(f"Sampled Data Shape: {df_sample.shape}")

In [ ]:
# 6. Save Final Dataset
output_name = 'processed_cicids2017.csv'
df_sample.to_csv(output_name, index=False)
print(f"Saved processed data to {output_name}")

# 7. Final Verification
print(f"Final Dataset Shape: {df_sample.shape}")
print("Final Label Value Counts:")
print(df_sample['Label'].value_counts())

# Metrics Report
end_time = time.time()
duration = end_time - start_time
feature_count = df_sample.shape[1] - 1 # Exclude Label

print("\n=== Preprocessing Metrics ===")
print(f"Computational Cost (Runtime): {duration:.4f} seconds")
print(f"Total Features Retained: {feature_count}")
print("(Note: 1 Duplicate Column was removed from original 78 features)")